# 🚀 DCT-GAN Mobile - Entrenamiento en GPU Cloud

**Replicación del paper**: *DCT-based Steganography using GAN with Mobile Architecture*

## 📋 Instrucciones

### Google Colab (Gratis con GPU T4):
1. **Runtime** → **Change runtime type** → **T4 GPU**
2. Ejecutar todas las celdas (**Runtime** → **Run all**)
3. Tiempo estimado: **1-2 días** (vs 19 días en CPU)

### Kaggle (Gratis 30h/semana GPU P100):
1. Upload este notebook
2. **Settings** → **Accelerator** → **GPU P100**
3. **Run All**

### RunPod / Vast.ai (GPU A100 - Rápido):
1. Rentar pod con Jupyter
2. Upload notebook
3. Tiempo: **~12 horas**

## 1️⃣ Setup Inicial - Verificar GPU

In [ ]:
import torch
import sys

print("="*70)
print("VERIFICANDO GPU")
print("="*70)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Disponible: {gpu_name}")
    print(f"✅ Memoria GPU: {gpu_memory:.1f} GB")
    print(f"✅ CUDA Version: {torch.version.cuda}")
    print(f"✅ PyTorch Version: {torch.__version__}")
else:
    print("❌ NO HAY GPU - El entrenamiento será MUY lento")
    print("   Cambia a GPU: Runtime → Change runtime type → GPU")

print(f"\n📍 Python: {sys.version}")
print("="*70)

## 2️⃣ Clonar Repositorio

In [ ]:
import os

# Si ya existe, eliminar
if os.path.exists('DCT-GAN-Mobile'):
    !rm -rf DCT-GAN-Mobile

# Clonar repositorio desde GitHub
!git clone https://github.com/jaimelopezm-star/DCT-GAN-Mobile.git

%cd DCT-GAN-Mobile

print("✅ Repositorio clonado exitosamente")
print("📂 Archivos disponibles:")
!ls -la

## 3️⃣ Subir Código

**TEMPORAL**: Sube estos archivos manualmente al notebook:
- `train.py`
- `models/` (carpeta completa)
- `configs/base_config.yaml`
- `utils/` (carpeta completa)

**MEJOR**: Una vez que subas a GitHub, el paso 2 clonará automáticamente.

In [ ]:
# Verificar archivos necesarios
import os

required_files = [
    'train.py',
    'models/encoder.py',
    'models/decoder.py',
    'models/discriminator.py',
    'configs/base_config.yaml',
    'utils/trainer.py',
    'utils/metrics.py'
]

print("Verificando archivos...")
missing = []
for f in required_files:
    if os.path.exists(f):
        print(f"✅ {f}")
    else:
        print(f"❌ {f} - FALTA")
        missing.append(f)

if missing:
    print(f"\n⚠️  Faltan {len(missing)} archivos. Súbelos manualmente o clona desde GitHub.")
else:
    print("\n✅ Todos los archivos presentes")

## 4️⃣ Instalar Dependencias

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pillow pyyaml tqdm tensorboard

print("✅ Dependencias instaladas")

## 5️⃣ Descargar Tiny ImageNet

**200 MB** - Descarga rápida (~2 minutos)

In [ ]:
import urllib.request
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm

print("="*70)
print("DESCARGANDO TINY IMAGENET (200 MB)")
print("="*70)

# Crear directorio
data_dir = Path("data/imagenet2012")
data_dir.mkdir(parents=True, exist_ok=True)

# Descargar
tiny_url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
tiny_zip = data_dir / "tiny-imagenet-200.zip"

print(f"\n📥 Descargando desde Stanford...")

class DownloadProgress(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

with DownloadProgress(unit='B', unit_scale=True, miniters=1, desc='Descarga') as t:
    urllib.request.urlretrieve(tiny_url, tiny_zip, reporthook=t.update_to)

print("\n✅ Descarga completada")

# Extraer
print("\n📦 Extrayendo...")
with zipfile.ZipFile(tiny_zip, 'r') as zip_ref:
    for member in tqdm(zip_ref.namelist(), desc="Extrayendo"):
        zip_ref.extract(member, data_dir)

print("✅ Extracción completada")

# Limpiar
tiny_zip.unlink()

# Organizar
print("\n📁 Organizando imágenes...")
tiny_dir = data_dir / "tiny-imagenet-200"
val_src = tiny_dir / "val" / "images"
organized_dir = data_dir / "organized" / "all"
organized_dir.mkdir(parents=True, exist_ok=True)

if val_src.exists():
    for img in tqdm(list(val_src.glob("*.JPEG")), desc="Copiando"):
        shutil.copy2(img, organized_dir / img.name)
    
    count = len(list(organized_dir.glob("*.JPEG")))
    print(f"\n✅ {count} imágenes organizadas")
    print(f"📍 Ubicación: {organized_dir.absolute()}")
else:
    print(f"❌ Error: No se encontró {val_src}")

## 6️⃣ Crear Splits (Train/Val/Test)

In [ ]:
import random

print("="*70)
print("CREANDO SPLITS")
print("="*70)

source_dir = Path("data/imagenet2012/organized/all")
base_dir = Path("data/imagenet2012/splits")

# Obtener imágenes
images = list(source_dir.glob("*.JPEG"))
print(f"\n📊 Total imágenes: {len(images)}")

# Shuffle
random.seed(42)
random.shuffle(images)

# Split 80/10/10
total = len(images)
train_size = int(0.8 * total)
val_size = int(0.1 * total)

train_imgs = images[:train_size]
val_imgs = images[train_size:train_size+val_size]
test_imgs = images[train_size+val_size:]

print(f"\nSplits:")
print(f"  Train: {len(train_imgs)} (80%)")
print(f"  Val:   {len(val_imgs)} (10%)")
print(f"  Test:  {len(test_imgs)} (10%)")

# Crear directorios
for split in ['train', 'val', 'test']:
    (base_dir / split / 'all').mkdir(parents=True, exist_ok=True)

# Copiar
print("\n⏳ Copiando imágenes...")

for img in tqdm(train_imgs, desc="Train"):
    shutil.copy2(img, base_dir / 'train' / 'all' / img.name)

for img in tqdm(val_imgs, desc="Val"):
    shutil.copy2(img, base_dir / 'val' / 'all' / img.name)

for img in tqdm(test_imgs, desc="Test"):
    shutil.copy2(img, base_dir / 'test' / 'all' / img.name)

print("\n✅ Splits creados")
print(f"📍 Ubicación: {base_dir.absolute()}")

## 7️⃣ Montar Google Drive (Opcional - Para guardar checkpoints)

**Recomendado**: Guarda los modelos en Drive para no perderlos si se cierra la sesión

In [ ]:
# Solo para Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Crear carpeta para checkpoints
    checkpoint_dir = '/content/drive/MyDrive/DCT-GAN-Checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    print(f"✅ Drive montado")
    print(f"📍 Checkpoints se guardarán en: {checkpoint_dir}")
    
    # Crear symlink
    if os.path.exists('checkpoints'):
        !rm -rf checkpoints
    !ln -s "{checkpoint_dir}" checkpoints
    
except ImportError:
    print("ℹ️  No estás en Colab - usando almacenamiento local")
    checkpoint_dir = 'checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)

## 8️⃣ ENTRENAR - 100 Épocas

**Tiempo estimado por GPU**:
- **T4 (Colab gratis)**: ~1-2 días (~2-3 min/época)
- **P100 (Kaggle)**: ~1 día (~1-2 min/época)
- **A100 (RunPod)**: ~12 horas (~30 seg/época)

**Objetivo**: PSNR ~52-55 dB

In [ ]:
# Entrenar
!python train.py \
    --config configs/base_config.yaml \
    --dataset imagenet \
    --checkpoint_dir checkpoints \
    --log_dir logs

## 9️⃣ Monitorear con TensorBoard

In [ ]:
# Cargar TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs

## 🔟 Descargar Checkpoints

Ejecuta esto para descargar el mejor modelo

In [ ]:
from google.colab import files
import shutil

# Comprimir checkpoints
!zip -r checkpoints_final.zip checkpoints/

# Descargar
files.download('checkpoints_final.zip')

print("✅ Checkpoints descargados")

## 📊 Resumen de Resultados

In [ ]:
# Mostrar métricas finales
import yaml

# Leer últimos logs
log_file = sorted(Path('logs').glob('*.log'))[-1]

print("="*70)
print("RESULTADOS FINALES")
print("="*70)

# Parsear últimas líneas
with open(log_file, 'r') as f:
    lines = f.readlines()[-50:]  # Últimas 50 líneas
    
for line in lines:
    if 'PSNR' in line or 'SSIM' in line or 'Best' in line:
        print(line.strip())

print("\n✅ Entrenamiento completado")
print("📍 Checkpoints guardados en Drive")